# 02_eval_parser_adapter_stratified

Eval-only notebook cho Government AI Parser.

Bản này giữ nguyên luồng evaluate/report, nhưng thay inference bằng catalog grounding + guard rails Phase 3/4:

1. Detect dataset + final LoRA adapter
2. Load Qwen3-4B + adapter
3. Stratified sample validation/test
4. Build candidates từ catalog cho từng user message
5. Strict prompt inference
6. Schema/catalog/evidence guard trước khi tính metrics
7. Xuất reports kiểm tra


In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers>=4.51.0",
    "accelerate",
    "peft",
    "bitsandbytes",
    "jsonschema",
    "protobuf>=5.29.1,<6",
    "sentencepiece",
])

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "wandb"], check=False)

import gc
import time
import math
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
import json

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from jsonschema import Draft7Validator

print("Setup OK", flush=True)
print("CUDA:", torch.cuda.is_available(), "GPUs:", torch.cuda.device_count(), flush=True)

In [ ]:
BASE_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

EVAL_MAX_SAMPLES_PER_SPLIT = 1000
EVAL_BATCH_SIZE = 8
MAX_NEW_TOKENS = 192
RANDOM_SEED = 42

KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/government-parser-eval")

RAW_DATA_DIR = WORK_DIR / "data" / "raw"
CONFIG_DIR = WORK_DIR / "configs"
MODEL_DIR = WORK_DIR / "model"
REPORT_DIR = WORK_DIR / "reports"
EXPORT_DIR = WORK_DIR / "export"

for p in [RAW_DATA_DIR, CONFIG_DIR, MODEL_DIR, REPORT_DIR, EXPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

def find_first_file(filename: str):
    matches = list(KAGGLE_INPUT_DIR.rglob(filename))
    return matches[0] if matches else None

def find_dataset_root():
    candidates = []

    for validation_path in KAGGLE_INPUT_DIR.rglob("parser_validation.v1.jsonl"):
        root = validation_path.parent
        score = 0

        if "government-parser-dataset-v1" in str(root):
            score += 100
        if (root / "parsed_query_schema.v1.json").exists():
            score += 50
        if (root / "parser_intents.v1.json").exists():
            score += 10
        if (root / "question_families.v1.json").exists():
            score += 10

        if "/notebooks/" in str(root):
            score -= 20

        candidates.append((score, root))

    if not candidates:
        raise FileNotFoundError("Không tìm thấy parser_validation.v1.jsonl trong /kaggle/input")

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    print("Dataset root candidates:")
    for score, root in candidates:
        print(score, root)

    return candidates[0][1]

SOURCE_DATASET_DIR = find_dataset_root()

validation_src = SOURCE_DATASET_DIR / "parser_validation.v1.jsonl"
test_src = SOURCE_DATASET_DIR / "parser_test.v1.jsonl"
schema_src = SOURCE_DATASET_DIR / "parsed_query_schema.v1.json"

print("Selected SOURCE_DATASET_DIR:", SOURCE_DATASET_DIR)

if validation_src is None or test_src is None or schema_src is None:
    raise FileNotFoundError("Thiếu dataset/config gốc. Hãy attach government-parser-dataset-v1 vào notebook.")

SOURCE_DATASET_DIR = validation_src.parent

required_data = ["parser_validation.v1.jsonl", "parser_test.v1.jsonl"]
required_configs = [
    "parsed_query_schema.v1.json",
    "parser_intents.v1.json",
    "parser_enums.v1.json",
    "question_families.v1.json",
    "country_catalog.v1.json",
    "indicator_catalog.v1.json",
    "analytics_metadata.v1.json",
]

for fn in required_data:
    shutil.copy2(SOURCE_DATASET_DIR / fn, RAW_DATA_DIR / fn)

for fn in required_configs:
    src = SOURCE_DATASET_DIR / fn
    if src.exists():
        shutil.copy2(src, CONFIG_DIR / fn)
    else:
        print("Warning missing config:", src, flush=True)

RAW_VALIDATION_PATH = RAW_DATA_DIR / "parser_validation.v1.jsonl"
RAW_TEST_PATH = RAW_DATA_DIR / "parser_test.v1.jsonl"

SCHEMA_PATH = CONFIG_DIR / "parsed_query_schema.v1.json"
INTENTS_PATH = CONFIG_DIR / "parser_intents.v1.json"
ENUMS_PATH = CONFIG_DIR / "parser_enums.v1.json"
QUESTION_FAMILIES_PATH = CONFIG_DIR / "question_families.v1.json"
COUNTRY_CATALOG_PATH = CONFIG_DIR / "country_catalog.v1.json"
INDICATOR_CATALOG_PATH = CONFIG_DIR / "indicator_catalog.v1.json"

def find_adapter_dirs():
    out = []
    for cfg in KAGGLE_INPUT_DIR.rglob("adapter_config.json"):
        d = cfg.parent
        if (d / "adapter_model.safetensors").exists():
            out.append(d)
    return out

adapter_dirs = find_adapter_dirs()
FINAL_ADAPTER_DIR = MODEL_DIR / "final_adapter"

if FINAL_ADAPTER_DIR.exists():
    shutil.rmtree(FINAL_ADAPTER_DIR)

if adapter_dirs:
    # Ưu tiên path có final_adapter/parser_qwen nếu có.
    adapter_dirs = sorted(adapter_dirs, key=lambda p: (("final" not in str(p).lower()) and ("adapter" not in str(p).lower()), len(str(p))))
    SOURCE_ADAPTER_DIR = adapter_dirs[0]
    shutil.copytree(SOURCE_ADAPTER_DIR, FINAL_ADAPTER_DIR)
else:
    zip_candidates = list(KAGGLE_INPUT_DIR.rglob("government_parser_qwen3_4b_lora_artifact.zip"))
    if not zip_candidates:
        raise FileNotFoundError("Không tìm thấy final adapter folder hoặc artifact zip trong /kaggle/input.")
    artifact_zip = zip_candidates[0]
    extract_dir = MODEL_DIR / "artifact_extract"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(artifact_zip, "r") as z:
        z.extractall(extract_dir)
    extracted = []
    for cfg in extract_dir.rglob("adapter_config.json"):
        d = cfg.parent
        if (d / "adapter_model.safetensors").exists():
            extracted.append(d)
    if not extracted:
        raise FileNotFoundError("Artifact zip không chứa adapter hợp lệ.")
    SOURCE_ADAPTER_DIR = extracted[0]
    shutil.copytree(SOURCE_ADAPTER_DIR, FINAL_ADAPTER_DIR)

print("SOURCE_DATASET_DIR:", SOURCE_DATASET_DIR, flush=True)
print("SOURCE_ADAPTER_DIR:", SOURCE_ADAPTER_DIR, flush=True)
print("FINAL_ADAPTER_DIR:", FINAL_ADAPTER_DIR, flush=True)
print("Adapter files:", sorted([p.name for p in FINAL_ADAPTER_DIR.iterdir()]), flush=True)

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

SCHEMA = load_json(SCHEMA_PATH)
INTENTS = set(load_json(INTENTS_PATH))
ENUMS = load_json(ENUMS_PATH)
QUESTION_FAMILIES = load_json(QUESTION_FAMILIES_PATH)
COUNTRY_CATALOG = load_json(COUNTRY_CATALOG_PATH)
INDICATOR_CATALOG = load_json(INDICATOR_CATALOG_PATH)

FAMILY_TO_INTENT = {x["id"]: x["intent"] for x in QUESTION_FAMILIES["families"]}
COUNTRY_CODES = {x["code"] for x in COUNTRY_CATALOG["countries"]}
INDICATOR_CODES = {x["code"] for x in INDICATOR_CATALOG["indicators"]}
ALLOWED_GROUPS = set(ENUMS.get("country_groups", []))

validator = Draft7Validator(SCHEMA)

validation_rows = load_jsonl(RAW_VALIDATION_PATH)
test_rows = load_jsonl(RAW_TEST_PATH)

print("Validation rows:", len(validation_rows), flush=True)
print("Test rows:", len(test_rows), flush=True)
print("Intents:", len(INTENTS), "Families:", len(FAMILY_TO_INTENT), flush=True)
print("Countries:", len(COUNTRY_CODES), "Indicators:", len(INDICATOR_CODES), flush=True)

In [ ]:
def stratified_sample(rows, n=None, seed=42):
    if n is None or n >= len(rows):
        return list(rows)

    rng = random.Random(seed)

    groups = defaultdict(list)
    for row in rows:
        key = (
            row.get("generation_source"),
            row.get("intent"),
            row.get("language_style"),
        )
        groups[key].append(row)

    selected = []
    group_items = list(groups.items())
    rng.shuffle(group_items)

    for key, items in group_items:
        if len(selected) < n:
            selected.append(rng.choice(items))

    remaining = n - len(selected)
    if remaining > 0:
        # proportional pool
        pool = []
        for key, items in group_items:
            items_copy = list(items)
            rng.shuffle(items_copy)
            pool.extend(items_copy)
        rng.shuffle(pool)

        seen_ids = {x.get("sample_id") for x in selected}
        for item in pool:
            if len(selected) >= n:
                break
            sid = item.get("sample_id")
            if sid not in seen_ids:
                selected.append(item)
                seen_ids.add(sid)

    rng.shuffle(selected)
    return selected[:n]

eval_validation_rows = stratified_sample(validation_rows, EVAL_MAX_SAMPLES_PER_SPLIT, RANDOM_SEED)
eval_test_rows = stratified_sample(test_rows, EVAL_MAX_SAMPLES_PER_SPLIT, RANDOM_SEED + 1)

def summarize_rows(name, rows):
    print("=" * 80, flush=True)
    print(name, "rows:", len(rows), flush=True)
    print("generation_source:", dict(Counter(r.get("generation_source") for r in rows)), flush=True)
    print("language_style:", dict(Counter(r.get("language_style") for r in rows)), flush=True)
    print("top intents:", Counter(r.get("intent") for r in rows).most_common(10), flush=True)

summarize_rows("validation_sample", eval_validation_rows)
summarize_rows("test_sample", eval_test_rows)

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

tokenizer = AutoTokenizer.from_pretrained(
    str(FINAL_ADAPTER_DIR),
    use_fast=True,
    trust_remote_code=False,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("[START] Load base model", flush=True)
t0 = time.time()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
    attn_implementation="sdpa",
)

print("[DONE] Load base model seconds:", round(time.time() - t0, 2), flush=True)

print("[START] Load LoRA adapter", flush=True)
model = PeftModel.from_pretrained(base_model, str(FINAL_ADAPTER_DIR))
model.eval()
model.config.use_cache = True

try:
    model.gradient_checkpointing_disable()
except Exception:
    pass

print("[DONE] Model ready", flush=True)

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: allocated={torch.cuda.memory_allocated(i)/1024**3:.3f}GB "
        f"reserved={torch.cuda.memory_reserved(i)/1024**3:.3f}GB",
        flush=True,
    )

In [ ]:
import re
import unicodedata
from typing import Any

INFERENCE_MODE = "catalog_grounding_guardrails_phase4"
MAX_INPUT_TOKENS = globals().get("MAX_INPUT_TOKENS", 1024)


def safe_json_loads(text: str):
    text = (text or "").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def validate_parsed_query(parsed: dict):
    errors = []

    for e in sorted(validator.iter_errors(parsed), key=lambda x: x.path):
        path = ".".join(str(p) for p in e.path)
        errors.append(f"schema_error:{path}:{e.message}")

    intent = parsed.get("intent")
    family = parsed.get("question_family")

    if intent not in INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if family not in FAMILY_TO_INTENT:
        errors.append(f"invalid_question_family:{family}")
    elif intent != FAMILY_TO_INTENT[family]:
        errors.append(f"family_intent_mismatch:{family}:parsed={intent}:expected={FAMILY_TO_INTENT[family]}")

    for x in parsed.get("indicators", []) or []:
        if x not in INDICATOR_CODES:
            errors.append(f"invalid_indicator:{x}")

    for x in parsed.get("countries", []) or []:
        if x not in COUNTRY_CODES:
            errors.append(f"invalid_country:{x}")

    for x in parsed.get("country_groups", []) or []:
        if x not in ALLOWED_GROUPS:
            errors.append(f"invalid_country_group:{x}")

    for field in ["ranking_order", "chart_preference", "aggregation", "relative_time", "event_time"]:
        allowed = set(ENUMS.get(field, []))
        if parsed.get(field) not in allowed:
            errors.append(f"invalid_enum:{field}:{parsed.get(field)}")

    return errors


def schema_validate_full(parsed: dict):
    errors = []
    for e in sorted(validator.iter_errors(parsed), key=lambda x: x.path):
        path = ".".join(str(p) for p in e.path)
        errors.append(f"schema_error:{path}:{e.message}")
    return len(errors) == 0, errors


def list_micro_counts(pred, gold):
    pred = set(pred or [])
    gold = set(gold or [])
    return len(pred & gold), len(pred - gold), len(gold - pred)


def is_executable(parsed):
    if not parsed:
        return False
    if parsed.get("intent") in {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}:
        return False
    if parsed.get("needs_clarification") is True:
        return False
    return True


CRITICAL_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
]


def is_dangerous_wrong(pred, gold, schema_ok):
    if not schema_ok or not is_executable(pred):
        return False

    if not is_executable(gold):
        return True

    for field in CRITICAL_FIELDS:
        if pred.get(field) != gold.get(field):
            return True

    return False


DEFAULT_FALLBACK_PARSED = {
    "intent": "NEED_CLARIFICATION",
    "question_family": "missing_indicator",
    "indicators": [],
    "countries": [],
    "country_groups": [],
    "start_year": None,
    "end_year": None,
    "relative_time": None,
    "event_time": None,
    "ranking_order": None,
    "limit": None,
    "threshold": None,
    "aggregation": None,
    "chart_preference": "none",
    "needs_clarification": True,
    "clarification_questions": [
        "Bạn muốn phân tích chỉ số nào, quốc gia nào và trong giai đoạn nào?"
    ],
    "confidence": 0.0,
}


def build_fallback_parsed(reason: str) -> dict[str, Any]:
    parsed = dict(DEFAULT_FALLBACK_PARSED)
    parsed["clarification_questions"] = list(DEFAULT_FALLBACK_PARSED["clarification_questions"])
    if reason == "invalid_json":
        parsed["clarification_questions"] = [
            "Parser chưa trả JSON hợp lệ. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    elif reason == "schema_error":
        parsed["clarification_questions"] = [
            "Parser trả kết quả chưa đúng schema. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    return parsed


def normalize_text(text: str) -> str:
    text = str(text or "").lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9%\s/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def as_list(value: Any) -> list:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        return [value]
    return []


def normalize_indicator_catalog_for_grounding(raw: Any) -> dict[str, dict]:
    records = raw.get("indicators", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code") if isinstance(row, dict) else None
        if not code:
            continue

        aliases = []
        aliases.append(code)
        aliases.append(row.get("name") or code)
        aliases.extend(as_list(row.get("aliases")))

        hint = row.get("question_templates_hint") or {}
        if isinstance(hint, dict):
            aliases.extend([
                hint.get("vi"),
                hint.get("vi_no_diacritics"),
                hint.get("en"),
                hint.get("technical"),
            ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[str(code)] = {
            "code": str(code),
            "name": row.get("name") or str(code),
            "aliases": clean_aliases,
        }

    return catalog


def normalize_country_catalog_for_grounding(raw: Any) -> dict[str, dict]:
    records = raw.get("countries", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code") if isinstance(row, dict) else None
        if not code:
            continue

        code = str(code).upper()
        aliases = []
        aliases.append(row.get("name") or code)
        aliases.extend(as_list(row.get("aliases")))

        hint = row.get("question_templates_hint") or {}
        if isinstance(hint, dict):
            aliases.extend([
                hint.get("vi"),
                hint.get("vi_no_diacritics"),
                hint.get("en"),
                hint.get("iso3"),
            ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[code] = {
            "code": code,
            "name": row.get("name") or code,
            "aliases": clean_aliases,
        }

    return catalog


INDICATOR_CATALOG_BY_CODE = normalize_indicator_catalog_for_grounding(INDICATOR_CATALOG)
COUNTRY_CATALOG_BY_CODE = normalize_country_catalog_for_grounding(COUNTRY_CATALOG)
ALLOWED_INTENTS = set(INTENTS)
QUESTION_FAMILY_IDS = set(FAMILY_TO_INTENT.keys())
QUESTION_FAMILY_BY_INTENT = defaultdict(list)
for family_id, intent in FAMILY_TO_INTENT.items():
    QUESTION_FAMILY_BY_INTENT[intent].append(family_id)

NON_EXECUTABLE_INTENTS = {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}
EXECUTABLE_INTENTS = set(ALLOWED_INTENTS) - NON_EXECUTABLE_INTENTS

REQUIRES_INDICATOR_INTENTS = {
    "COMPARE_COUNTRIES",
    "COMPARE_INDICATORS",
    "RANKING",
    "RANK_BY_CHANGE",
    "TIME_SERIES",
    "VALUE_LOOKUP",
    "TREND_ANALYSIS",
    "ANOMALY_DETECTION",
    "VISUALIZATION",
} & ALLOWED_INTENTS

REQUIRES_TIME_INTENTS = {
    "VALUE_LOOKUP",
    "RANKING",
    "COMPARE_COUNTRIES",
    "TIME_SERIES",
    "TREND_ANALYSIS",
    "RANK_BY_CHANGE",
    "ANOMALY_DETECTION",
    "VISUALIZATION",
} & ALLOWED_INTENTS


def alias_in_text(normalized_text: str, alias: str) -> bool:
    normalized_alias = normalize_text(alias)
    if not normalized_alias:
        return False

    if len(normalized_alias) <= 3:
        return re.search(rf"(^|\s){re.escape(normalized_alias)}($|\s)", normalized_text) is not None

    return normalized_alias in normalized_text


def score_alias(normalized_text: str, alias: str) -> float:
    normalized_alias = normalize_text(alias)
    if not alias_in_text(normalized_text, alias):
        return 0.0

    if normalized_text == normalized_alias:
        return 1.0

    return min(0.99, 0.55 + len(normalized_alias) / 60)


def resolve_candidate_indicators(message: str, limit: int = 8) -> list[dict]:
    normalized_message = normalize_text(message)
    matches = []

    for code, meta in INDICATOR_CATALOG_BY_CODE.items():
        best_score = 0.0
        best_alias = ""

        aliases = []
        aliases.append(code)
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        for alias in aliases:
            score = score_alias(normalized_message, alias)
            if score > best_score:
                best_score = score
                best_alias = alias

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]


def resolve_candidate_countries(message: str, limit: int = 12) -> list[dict]:
    normalized_message = normalize_text(message)
    raw_message = str(message or "")
    matches = []

    raw_tokens = set(re.findall(r"\b[A-Z]{3}\b", raw_message))

    for code, meta in COUNTRY_CATALOG_BY_CODE.items():
        best_score = 0.0
        best_alias = ""

        aliases = []
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        if code in raw_tokens:
            aliases.append(code)

        for alias in aliases:
            alias_text = str(alias or "").strip()

            if alias_text.upper() == code and code not in raw_tokens:
                continue

            score = score_alias(normalized_message, alias_text)
            if score > best_score:
                best_score = score
                best_alias = alias_text

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]


def resolve_candidate_country_groups(message: str, limit: int = 8) -> list[dict]:
    normalized_message = normalize_text(message)
    matches = []

    group_aliases = {
        "ASEAN": ["asean", "dong nam a", "đông nam á", "southeast asia"],
        "EU": ["eu", "european union", "lien minh chau au", "liên minh châu âu"],
        "OECD": ["oecd"],
        "G7": ["g7"],
        "G20": ["g20"],
    }

    for group in sorted(ALLOWED_GROUPS):
        aliases = [group, str(group).replace("_", " ")]
        aliases.extend(group_aliases.get(str(group).upper(), []))

        best_score = 0.0
        best_alias = ""
        for alias in aliases:
            score = score_alias(normalized_message, alias)
            if score > best_score:
                best_score = score
                best_alias = alias

        if best_score >= 0.6:
            matches.append({
                "code": group,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)
    return matches[:limit]


def resolve_candidate_years(message: str) -> list[int]:
    years = []
    for match in re.finditer(r"\b(19\d{2}|20\d{2}|21\d{2})\b", str(message or "")):
        year = int(match.group(1))
        if 1900 <= year <= 2199 and year not in years:
            years.append(year)
    return years


def add_families_for_intent(families: list[str], intent: str):
    for family in QUESTION_FAMILY_BY_INTENT.get(intent, []):
        if family not in families:
            families.append(family)


def infer_candidate_intents_and_families(message: str, indicators: list, countries: list, country_groups: list, years: list) -> tuple[list[str], list[str]]:
    normalized = normalize_text(message)
    intents = []
    families = []

    def add_intent(intent: str):
        if intent in ALLOWED_INTENTS and intent not in intents:
            intents.append(intent)
            add_families_for_intent(families, intent)

    if any(token in normalized for token in ["viet sql", "write sql", "raw sql", "cau lenh sql", "lệnh sql"]):
        add_intent("UNSUPPORTED")
    elif any(token in normalized for token in ["du bao", "forecast", "arima", "predict", "prediction", "mo hinh", "mô hình"]):
        add_intent("UNSUPPORTED")
    elif any(token in normalized for token in ["so sanh", "compare", "doi chieu", "đối chiếu", "voi", "với"]):
        add_intent("COMPARE_COUNTRIES")
    elif any(token in normalized for token in ["top", "cao nhat", "cao nhất", "thap nhat", "thấp nhất", "xep hang", "xếp hạng", "ranking", "nuoc nao", "nước nào"]):
        add_intent("RANKING")
    elif any(token in normalized for token in ["xu huong", "xu hướng", "trend", "dien bien", "diễn biến"]):
        add_intent("TREND_ANALYSIS")
        add_intent("TIME_SERIES")
    elif any(token in normalized for token in ["ve", "vẽ", "chart", "bieu do", "biểu đồ", "line chart", "bar chart"]):
        add_intent("VISUALIZATION")
        add_intent("TIME_SERIES")
    elif any(token in normalized for token in ["du lieu", "dữ liệu", "coverage", "bao phu", "bao phủ", "thieu du lieu", "thiếu dữ liệu"]):
        add_intent("COVERAGE")
    elif indicators and (countries or country_groups) and years:
        add_intent("VALUE_LOOKUP")
    elif indicators and (countries or country_groups):
        add_intent("NEED_CLARIFICATION")
    elif countries or country_groups:
        add_intent("NEED_CLARIFICATION")

    if not intents:
        add_intent("NEED_CLARIFICATION")

    return intents, families[:16]


def build_grounding_candidates(message: str, context: dict | None = None) -> dict[str, Any]:
    candidate_indicators = resolve_candidate_indicators(message)
    candidate_countries = resolve_candidate_countries(message)
    candidate_country_groups = resolve_candidate_country_groups(message)
    detected_years = resolve_candidate_years(message)
    candidate_intents, candidate_question_families = infer_candidate_intents_and_families(
        message,
        candidate_indicators,
        candidate_countries,
        candidate_country_groups,
        detected_years,
    )

    return {
        "candidate_indicators": candidate_indicators,
        "candidate_countries": candidate_countries,
        "candidate_country_groups": candidate_country_groups,
        "detected_years": detected_years,
        "candidate_intents": candidate_intents,
        "candidate_question_families": candidate_question_families,
    }


def compact_candidates_for_prompt(candidates: dict[str, Any]) -> dict[str, Any]:
    return {
        "candidate_indicators": [
            {
                "code": item.get("code"),
                "name": item.get("name"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_indicators", [])
        ],
        "candidate_countries": [
            {
                "code": item.get("code"),
                "name": item.get("name"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_countries", [])
        ],
        "candidate_country_groups": [
            {
                "code": item.get("code"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_country_groups", [])
        ],
        "detected_years": candidates.get("detected_years", []),
        "candidate_intents": candidates.get("candidate_intents", []),
        "candidate_question_families": candidates.get("candidate_question_families", []),
    }


def build_parser_messages(message: str, context: dict | None = None, candidates: dict | None = None) -> list[dict[str, str]]:
    candidates = candidates or build_grounding_candidates(message, context)

    system_prompt = """
You are a semantic parser for a Government Economic Indicator AI Agent.
Return only one valid JSON object matching ParsedQuery v1.
Do not answer the question.
Do not write SQL.
Do not query data.
Do not invent missing indicators, countries, country groups, years, or question families.
Prefer NEED_CLARIFICATION over guessing.
Only use valid intent, question_family, indicator code, country code, and enum values from the provided catalogs/candidates.
If the user asks for forecasting, ARIMA/modeling, raw SQL, causal claims, or unsupported operations, return UNSUPPORTED.
""".strip()

    user_payload = {
        "message": message,
        "context": context,
        "grounding_candidates": compact_candidates_for_prompt(candidates),
        "required_output_fields": [
            "intent",
            "question_family",
            "indicators",
            "countries",
            "country_groups",
            "start_year",
            "end_year",
            "relative_time",
            "event_time",
            "ranking_order",
            "limit",
            "threshold",
            "aggregation",
            "chart_preference",
            "needs_clarification",
            "clarification_questions",
            "confidence",
        ],
    }

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
    ]


def safe_list(value: Any) -> list:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def candidate_codes(candidates: dict | None, key: str) -> set[str]:
    values = set()
    items = (candidates or {}).get(key) or []

    for item in items:
        if isinstance(item, dict):
            code = item.get("code")
            if code:
                values.add(str(code))
        elif item is not None:
            values.add(str(item))

    return values


def detected_year_set(candidates: dict | None) -> set[int]:
    result = set()
    for item in (candidates or {}).get("detected_years") or []:
        try:
            result.add(int(item))
        except Exception:
            pass
    return result


def normalize_parsed_basic(parsed: dict) -> dict:
    normalized = dict(DEFAULT_FALLBACK_PARSED)
    normalized.update(parsed or {})

    normalized["intent"] = str(normalized.get("intent") or "").strip()
    normalized["question_family"] = str(normalized.get("question_family") or "").strip()

    normalized["indicators"] = [
        str(item).strip()
        for item in safe_list(normalized.get("indicators"))
        if item is not None and str(item).strip()
    ]

    normalized["countries"] = [
        str(item).strip().upper()
        for item in safe_list(normalized.get("countries"))
        if item is not None and str(item).strip()
    ]

    normalized["country_groups"] = [
        str(item).strip()
        for item in safe_list(normalized.get("country_groups"))
        if item is not None and str(item).strip()
    ]

    normalized["clarification_questions"] = [
        str(item).strip()
        for item in safe_list(normalized.get("clarification_questions"))
        if item is not None and str(item).strip()
    ]

    for field in ["start_year", "end_year", "limit"]:
        value = normalized.get(field)
        if value is not None:
            try:
                normalized[field] = int(value)
            except Exception:
                pass

    value = normalized.get("confidence")
    try:
        normalized["confidence"] = float(value)
    except Exception:
        normalized["confidence"] = 0.0

    return normalized


def family_allowed_for_intent(question_family: str, intent: str) -> bool:
    if not question_family or not intent:
        return False

    if question_family not in QUESTION_FAMILY_IDS:
        return False

    expected_intent = FAMILY_TO_INTENT.get(question_family)
    if expected_intent:
        return expected_intent == intent

    if intent == "NEED_CLARIFICATION":
        return question_family.startswith("missing_") or question_family.startswith("ambiguous_")

    if intent == "UNSUPPORTED":
        return question_family.startswith("unsupported_")

    return True


def derive_safe_question_family(parsed: dict, candidates: dict | None) -> str | None:
    intent = parsed.get("intent")

    def allowed(family: str) -> str | None:
        if family in QUESTION_FAMILY_IDS and family_allowed_for_intent(family, intent):
            return family
        return None

    if intent == "COMPARE_COUNTRIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")

        if len(countries) >= 2:
            if start_year is not None and end_year is not None and start_year == end_year:
                return allowed("compare_countries_single_year")
            if start_year is not None and end_year is not None:
                return allowed("compare_countries_period")

    if intent == "RANKING":
        order = parsed.get("ranking_order")
        if order == "desc":
            return allowed("ranking_top_n")
        if order == "asc":
            return allowed("ranking_bottom_n")

    if intent == "TREND_ANALYSIS":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("trend_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("trend_multi_country_period")

    if intent == "TIME_SERIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("time_series_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("time_series_multi_country_period")

    return None


def make_clarification(parsed: dict, question: str, family: str) -> dict:
    updated = dict(parsed)
    updated["intent"] = "NEED_CLARIFICATION"
    updated["question_family"] = family
    updated["needs_clarification"] = True
    updated["clarification_questions"] = [question]
    updated["confidence"] = min(float(updated.get("confidence") or 0.6), 0.6)
    return updated


def validate_catalog_and_guard(parsed: dict, message: str, candidates: dict | None = None) -> dict:
    errors = []
    repairs_applied = []

    normalized = normalize_parsed_basic(parsed)

    intent = normalized.get("intent")
    question_family = normalized.get("question_family")

    candidate_indicator_codes = candidate_codes(candidates, "candidate_indicators")
    candidate_country_codes = candidate_codes(candidates, "candidate_countries")
    candidate_group_codes = candidate_codes(candidates, "candidate_country_groups")
    years_in_user = detected_year_set(candidates)

    if intent not in ALLOWED_INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if question_family not in QUESTION_FAMILY_IDS:
        derived_family = derive_safe_question_family(normalized, candidates)
        if derived_family:
            repairs_applied.append(f"question_family:{question_family}->{derived_family}")
            normalized["question_family"] = derived_family
            question_family = derived_family
        else:
            errors.append(f"invalid_question_family:{question_family}")

    if question_family in QUESTION_FAMILY_IDS:
        if not family_allowed_for_intent(question_family, intent):
            derived_family = derive_safe_question_family(normalized, candidates)
            if derived_family:
                repairs_applied.append(f"question_family:{question_family}->{derived_family}")
                normalized["question_family"] = derived_family
                question_family = derived_family
            else:
                errors.append(f"question_family_not_allowed_for_intent:{question_family}:{intent}")

    for indicator in normalized.get("indicators", []):
        if indicator not in INDICATOR_CATALOG_BY_CODE:
            errors.append(f"invalid_indicator:{indicator}")
        elif candidate_indicator_codes and indicator not in candidate_indicator_codes:
            errors.append(f"indicator_not_mentioned_by_user:{indicator}")
        elif not candidate_indicator_codes:
            errors.append(f"indicator_without_user_evidence:{indicator}")

    for country in normalized.get("countries", []):
        if country not in COUNTRY_CATALOG_BY_CODE:
            errors.append(f"invalid_country:{country}")
        elif candidate_country_codes and country not in candidate_country_codes:
            errors.append(f"country_not_mentioned_by_user:{country}")
        elif not candidate_country_codes:
            errors.append(f"country_without_user_evidence:{country}")

    for group in normalized.get("country_groups", []):
        if group not in ALLOWED_GROUPS:
            errors.append(f"invalid_country_group:{group}")
        elif candidate_group_codes and group not in candidate_group_codes:
            errors.append(f"country_group_not_mentioned_by_user:{group}")
        elif not candidate_group_codes:
            errors.append(f"country_group_without_user_evidence:{group}")

    if intent == "UNSUPPORTED":
        normalized["needs_clarification"] = False
        normalized["clarification_questions"] = []
        return {
            "parsed": normalized,
            "catalog_pass": True,
            "safe_to_execute": False,
            "fallback_reason": "unsupported_request",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent == "OFF_TOPIC":
        normalized["needs_clarification"] = False
        normalized["clarification_questions"] = []
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "off_topic",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent == "NEED_CLARIFICATION" or normalized.get("needs_clarification") is True:
        normalized["intent"] = "NEED_CLARIFICATION"
        normalized["needs_clarification"] = True
        if not normalized.get("clarification_questions"):
            normalized["clarification_questions"] = [
                "Bạn muốn phân tích chỉ số nào, quốc gia nào và trong giai đoạn nào?"
            ]
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "needs_clarification",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent in REQUIRES_INDICATOR_INTENTS:
        if not normalized.get("indicators") or not candidate_indicator_codes:
            fallback = make_clarification(
                normalized,
                "Bạn muốn so sánh/phân tích chỉ số nào?",
                "missing_indicator",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_indicator_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

    if intent in REQUIRES_TIME_INTENTS:
        start_year = normalized.get("start_year")
        end_year = normalized.get("end_year")

        if start_year is None or end_year is None or not years_in_user:
            fallback = make_clarification(
                normalized,
                "Bạn muốn phân tích trong năm nào hoặc giai đoạn nào?",
                "ambiguous_time_range",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_time_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

        if isinstance(start_year, int) and start_year not in years_in_user:
            errors.append(f"start_year_not_mentioned_by_user:{start_year}")

        if isinstance(end_year, int) and end_year not in years_in_user:
            errors.append(f"end_year_not_mentioned_by_user:{end_year}")

    if errors:
        fallback = make_clarification(
            normalized,
            "Mình chưa xác thực được chỉ số, quốc gia hoặc thời gian trong câu hỏi. Bạn có thể nói rõ hơn không?",
            "ambiguous_indicator",
        )
        return {
            "parsed": fallback,
            "catalog_pass": False,
            "safe_to_execute": False,
            "fallback_reason": "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent not in EXECUTABLE_INTENTS:
        return {
            "parsed": normalized,
            "catalog_pass": True,
            "safe_to_execute": False,
            "fallback_reason": "non_executable_intent",
            "repairs_applied": repairs_applied,
            "validation_errors": [],
        }

    return {
        "parsed": normalized,
        "catalog_pass": True,
        "safe_to_execute": True,
        "fallback_reason": None,
        "repairs_applied": repairs_applied,
        "validation_errors": [],
    }


def finalize_decoded_output(decoded: str, message: str, candidates: dict[str, Any]) -> dict[str, Any]:
    raw_pred = None
    final_pred = None
    json_ok = False
    raw_schema_ok = False
    catalog_pass = False
    safe_to_execute = False
    fallback_reason = None
    repairs_applied = []
    errors = []

    try:
        raw_pred = safe_json_loads(decoded)
        json_ok = True
    except Exception as exc:
        errors.append(f"json_error:{str(exc)}")
        final_pred = build_fallback_parsed("invalid_json")
        fallback_reason = "invalid_json"

    if raw_pred is not None:
        raw_schema_ok, schema_errors = schema_validate_full(raw_pred)
        errors.extend(schema_errors)

        if not raw_schema_ok:
            final_pred = build_fallback_parsed("schema_error")
            fallback_reason = "schema_error"
        else:
            guard_result = validate_catalog_and_guard(raw_pred, message, candidates)
            final_pred = guard_result["parsed"]
            catalog_pass = guard_result["catalog_pass"]
            safe_to_execute = guard_result["safe_to_execute"]
            fallback_reason = guard_result["fallback_reason"]
            repairs_applied = guard_result["repairs_applied"]
            errors.extend(guard_result["validation_errors"])

    final_validation_errors = validate_parsed_query(final_pred) if final_pred is not None else []

    return {
        "raw_pred": raw_pred,
        "pred": final_pred,
        "json_ok": json_ok,
        "raw_schema_ok": raw_schema_ok,
        "schema_ok": len(final_validation_errors) == 0,
        "catalog_pass": catalog_pass,
        "safe_to_execute": safe_to_execute,
        "fallback_reason": fallback_reason,
        "repairs_applied": repairs_applied,
        "errors": errors,
        "final_validation_errors": final_validation_errors,
    }

print("Inference mode:", INFERENCE_MODE, flush=True)
print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS, flush=True)
print("Grounding catalogs:", len(INDICATOR_CATALOG_BY_CODE), "indicators,", len(COUNTRY_CATALOG_BY_CODE), "countries", flush=True)


In [ ]:
@torch.no_grad()
def generate_batch(prompt_messages_batch):
    prompts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=True,
        )
        for msgs in prompt_messages_batch
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
        add_special_tokens=False,
    )

    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}
    prompt_width = inputs["input_ids"].shape[1]

    generated = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    outputs = []
    for i in range(generated.shape[0]):
        new_tokens = generated[i][prompt_width:]
        outputs.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return outputs


def evaluate_rows(split_name, rows):
    report_path = REPORT_DIR / f"{split_name}_stratified_predictions.jsonl"
    metrics_path = REPORT_DIR / f"{split_name}_stratified_metrics.json"
    error_path = REPORT_DIR / f"{split_name}_stratified_error_examples.json"

    total = len(rows)
    counters = Counter()

    indicator_tp = indicator_fp = indicator_fn = 0
    country_tp = country_fp = country_fn = 0
    group_tp = group_fp = group_fn = 0

    confusion_intent = Counter()
    confusion_family = Counter()
    schema_error_top = Counter()
    fallback_reason_top = Counter()
    validation_error_top = Counter()
    dangerous_examples = []
    schema_examples = []
    mismatch_examples = []

    start_time = time.time()

    with report_path.open("w", encoding="utf-8") as f_out:
        for start in range(0, total, EVAL_BATCH_SIZE):
            batch = rows[start:start + EVAL_BATCH_SIZE]

            prompt_batch = []
            gold_batch = []
            candidates_batch = []

            for row in batch:
                user_message = row.get("user_message")
                if not user_message:
                    messages = row.get("messages") or []
                    user_message = messages[1]["content"] if len(messages) >= 2 else ""

                candidates = build_grounding_candidates(user_message)
                prompt = build_parser_messages(user_message, None, candidates)

                messages = row.get("messages")
                gold = row.get("assistant_json")
                if gold is None and messages and len(messages) >= 3:
                    gold = json.loads(messages[2]["content"])

                prompt_batch.append(prompt)
                gold_batch.append(gold)
                candidates_batch.append(candidates)

            decoded_batch = generate_batch(prompt_batch)

            for row, gold, decoded, candidates in zip(batch, gold_batch, decoded_batch, candidates_batch):
                user_message = row.get("user_message")
                if not user_message:
                    messages = row.get("messages") or []
                    user_message = messages[1]["content"] if len(messages) >= 2 else ""

                result = finalize_decoded_output(decoded, user_message, candidates)
                raw_pred = result["raw_pred"]
                pred = result["pred"]
                json_ok = result["json_ok"]
                schema_ok = result["schema_ok"]
                errors = list(result["errors"]) + list(result["final_validation_errors"])
                catalog_pass = result["catalog_pass"]
                safe_to_execute = result["safe_to_execute"]
                fallback_reason = result["fallback_reason"]
                repairs_applied = result["repairs_applied"]

                if json_ok:
                    counters["valid_json"] += 1

                if schema_ok:
                    counters["schema_pass"] += 1
                else:
                    for e in errors:
                        schema_error_top[e] += 1

                if catalog_pass:
                    counters["catalog_pass"] += 1

                if safe_to_execute:
                    counters["safe_to_execute"] += 1

                if fallback_reason:
                    fallback_reason_top[fallback_reason] += 1

                for e in errors:
                    validation_error_top[e] += 1

                if pred == gold:
                    counters["exact_full_json"] += 1

                if schema_ok:
                    if pred.get("intent") == gold.get("intent"):
                        counters["intent_correct"] += 1
                    else:
                        confusion_intent[(gold.get("intent"), pred.get("intent"))] += 1

                    if pred.get("question_family") == gold.get("question_family"):
                        counters["family_correct"] += 1
                    else:
                        confusion_family[(gold.get("question_family"), pred.get("question_family"))] += 1

                    if pred.get("start_year") == gold.get("start_year") and pred.get("end_year") == gold.get("end_year"):
                        counters["year_correct"] += 1

                    for field in ["ranking_order", "aggregation", "chart_preference", "needs_clarification"]:
                        if pred.get(field) == gold.get(field):
                            counters[f"{field}_correct"] += 1

                    tp, fp, fn = list_micro_counts(pred.get("indicators"), gold.get("indicators"))
                    indicator_tp += tp; indicator_fp += fp; indicator_fn += fn

                    tp, fp, fn = list_micro_counts(pred.get("countries"), gold.get("countries"))
                    country_tp += tp; country_fp += fp; country_fn += fn

                    tp, fp, fn = list_micro_counts(pred.get("country_groups"), gold.get("country_groups"))
                    group_tp += tp; group_fp += fp; group_fn += fn

                if is_executable(pred):
                    counters["executable_plan"] += 1

                dangerous = is_dangerous_wrong(pred, gold, schema_ok)
                if dangerous:
                    counters["dangerous_wrong"] += 1
                    if len(dangerous_examples) < 50:
                        dangerous_examples.append({
                            "sample_id": row.get("sample_id"),
                            "generation_source": row.get("generation_source"),
                            "language_style": row.get("language_style"),
                            "user_message": user_message,
                            "gold": gold,
                            "pred": pred,
                            "raw_pred": raw_pred,
                            "decoded": decoded,
                            "candidates": candidates,
                            "catalog_pass": catalog_pass,
                            "safe_to_execute": safe_to_execute,
                            "fallback_reason": fallback_reason,
                            "validation_errors": errors,
                        })

                if errors and len(schema_examples) < 30:
                    schema_examples.append({
                        "sample_id": row.get("sample_id"),
                        "user_message": user_message,
                        "errors": errors,
                        "decoded": decoded,
                        "gold": gold,
                        "raw_pred": raw_pred,
                        "pred": pred,
                        "candidates": candidates,
                        "catalog_pass": catalog_pass,
                        "safe_to_execute": safe_to_execute,
                        "fallback_reason": fallback_reason,
                    })

                if schema_ok and pred != gold and len(mismatch_examples) < 50:
                    mismatch_examples.append({
                        "sample_id": row.get("sample_id"),
                        "generation_source": row.get("generation_source"),
                        "language_style": row.get("language_style"),
                        "user_message": user_message,
                        "gold": gold,
                        "pred": pred,
                        "raw_pred": raw_pred,
                        "candidates": candidates,
                        "catalog_pass": catalog_pass,
                        "safe_to_execute": safe_to_execute,
                        "fallback_reason": fallback_reason,
                    })

                f_out.write(json.dumps({
                    "sample_id": row.get("sample_id"),
                    "generation_source": row.get("generation_source"),
                    "language_style": row.get("language_style"),
                    "user_message": user_message,
                    "gold": gold,
                    "decoded": decoded,
                    "raw_pred": raw_pred,
                    "pred": pred,
                    "json_ok": json_ok,
                    "raw_schema_ok": result["raw_schema_ok"],
                    "schema_ok": schema_ok,
                    "catalog_pass": catalog_pass,
                    "safe_to_execute": safe_to_execute,
                    "fallback_reason": fallback_reason,
                    "repairs_applied": repairs_applied,
                    "candidates": candidates,
                    "errors": errors,
                }, ensure_ascii=False) + "\n")

            done = min(start + EVAL_BATCH_SIZE, total)
            if done % 100 == 0 or done == total:
                elapsed = time.time() - start_time
                speed = done / elapsed if elapsed else 0
                eta = (total - done) / speed if speed else 0
                print(f"{split_name}: {done}/{total} | {speed:.3f} samples/s | ETA {eta/60:.1f}m", flush=True)

    def rate(x):
        return x / total if total else 0

    schema_total = counters["schema_pass"] or 1

    def prf(tp, fp, fn):
        p = tp / (tp + fp) if (tp + fp) else 1.0
        r = tp / (tp + fn) if (tp + fn) else 1.0
        f1 = 2 * p * r / (p + r) if (p + r) else 0.0
        return {"precision": p, "recall": r, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

    metrics = {
        "split": split_name,
        "inference_mode": INFERENCE_MODE,
        "total": total,
        "valid_json_rate": rate(counters["valid_json"]),
        "schema_pass_rate": rate(counters["schema_pass"]),
        "catalog_pass_rate": rate(counters["catalog_pass"]),
        "safe_to_execute_rate": rate(counters["safe_to_execute"]),
        "exact_full_json_rate": rate(counters["exact_full_json"]),
        "intent_accuracy_on_schema_pass": counters["intent_correct"] / schema_total,
        "question_family_accuracy_on_schema_pass": counters["family_correct"] / schema_total,
        "year_exact_accuracy_on_schema_pass": counters["year_correct"] / schema_total,
        "ranking_order_accuracy_on_schema_pass": counters["ranking_order_correct"] / schema_total,
        "aggregation_accuracy_on_schema_pass": counters["aggregation_correct"] / schema_total,
        "chart_preference_accuracy_on_schema_pass": counters["chart_preference_correct"] / schema_total,
        "needs_clarification_accuracy_on_schema_pass": counters["needs_clarification_correct"] / schema_total,
        "indicator_micro": prf(indicator_tp, indicator_fp, indicator_fn),
        "country_micro": prf(country_tp, country_fp, country_fn),
        "country_group_micro": prf(group_tp, group_fp, group_fn),
        "executable_plan_rate": rate(counters["executable_plan"]),
        "dangerous_wrong_query_rate": rate(counters["dangerous_wrong"]),
        "dangerous_wrong_query_count": counters["dangerous_wrong"],
        "fallback_reason_top": dict(fallback_reason_top.most_common(30)),
        "schema_error_top": dict(schema_error_top.most_common(30)),
        "validation_error_top": dict(validation_error_top.most_common(30)),
        "intent_confusion_top": [
            {"gold": k[0], "pred": k[1], "count": v}
            for k, v in confusion_intent.most_common(30)
        ],
        "family_confusion_top": [
            {"gold": k[0], "pred": k[1], "count": v}
            for k, v in confusion_family.most_common(30)
        ],
        "runtime_seconds": round(time.time() - start_time, 2),
        "prediction_path": str(report_path),
        "metrics_path": str(metrics_path),
        "error_examples_path": str(error_path),
    }

    error_payload = {
        "dangerous_examples": dangerous_examples,
        "schema_examples": schema_examples,
        "mismatch_examples": mismatch_examples,
    }

    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    with error_path.open("w", encoding="utf-8") as f:
        json.dump(error_payload, f, ensure_ascii=False, indent=2)

    print("Saved:", metrics_path, flush=True)
    print(json.dumps(metrics, ensure_ascii=False, indent=2), flush=True)
    return metrics


validation_metrics = evaluate_rows("validation", eval_validation_rows)
test_metrics = evaluate_rows("test", eval_test_rows)


In [ ]:
SMOKE_MESSAGES = [
    "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023.",
    "Top 10 nước có lạm phát CPI cao nhất năm 2020.",
    "Xu hướng thất nghiệp của Nhật Bản sau COVID.",
    "Việt Nam có dữ liệu nợ công từ năm nào đến năm nào?",
    "Vẽ line chart cho tăng trưởng GDP thực của Indonesia giai đoạn 2000-2022.",
    "Nước nào trong ASEAN có tỷ lệ nghèo giảm mạnh nhất?",
    "Dự báo nợ công Việt Nam bằng ARIMA.",
    "Viết SQL lấy bảng gold_fiscal_monetary.",
    "Việt Nam thế nào?",
    "So sánh Việt Nam với Thái Lan.",
]

smoke_results = []
for msg in SMOKE_MESSAGES:
    candidates = build_grounding_candidates(msg)
    messages = build_parser_messages(msg, None, candidates)
    decoded = generate_batch([messages])[0]
    result = finalize_decoded_output(decoded, msg, candidates)

    smoke_results.append({
        "message": msg,
        "candidates": candidates,
        "raw_output": decoded,
        "raw_pred": result["raw_pred"],
        "pred": result["pred"],
        "valid_json": result["json_ok"],
        "raw_schema_pass": result["raw_schema_ok"],
        "schema_pass": result["schema_ok"],
        "catalog_pass": result["catalog_pass"],
        "safe_to_execute": result["safe_to_execute"],
        "fallback_reason": result["fallback_reason"],
        "repairs_applied": result["repairs_applied"],
        "errors": result["errors"] + result["final_validation_errors"],
    })

smoke_path = REPORT_DIR / "manual_smoke_test_results.json"
with smoke_path.open("w", encoding="utf-8") as f:
    json.dump(smoke_results, f, ensure_ascii=False, indent=2)

print("Saved:", smoke_path, flush=True)
print(json.dumps(smoke_results, ensure_ascii=False, indent=2), flush=True)


In [ ]:
summary = {
    "base_model_name": BASE_MODEL_NAME,
    "final_adapter_dir": str(FINAL_ADAPTER_DIR),
    "inference_mode": INFERENCE_MODE,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "eval_max_samples_per_split": EVAL_MAX_SAMPLES_PER_SPLIT,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
}

summary_path = REPORT_DIR / "eval_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

zip_path = EXPORT_DIR / "parser_eval_reports.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in REPORT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(REPORT_DIR))

print("REPORT_DIR:", REPORT_DIR, flush=True)
print("Eval report zip:", zip_path, flush=True)
print("Zip size MB:", round(zip_path.stat().st_size / 1024**2, 2), flush=True)

print("\nFiles to send/check:")
for p in sorted(REPORT_DIR.iterdir()):
    print("-", p.name, round(p.stat().st_size / 1024**2, 2), "MB", flush=True)
